In [ ]:
!pip install torch yfinance pandas matplotlib scikit-learn torchinfo scipy pytorch-wavelets

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from torchinfo import summary
from sklearn.preprocessing import StandardScaler
from scipy.stats import kurtosis, skew
from statsmodels.graphics.tsaplots import plot_acf

# ==========================================================
# АРХИТЕКТУРА MSLAD (Multi-Scale Latent Attention Diffusion)
# ==========================================================
class MSLAD_Core(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        
        self.encoder_fast = nn.Conv1d(1, 16, kernel_size=3, padding=1)
        self.encoder_slow = nn.Conv1d(1, 16, kernel_size=11, padding=5)
        
        
        self.attention = nn.MultiheadAttention(embed_dim=32, num_heads=4, batch_first=True)
        
        
        self.time_mlp = nn.Sequential(
            nn.Linear(1, 32), nn.SiLU(), nn.Linear(32, 32)
        )
        
        
        self.res_block = nn.Sequential(
            nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, 32)
        )
        self.decoder = nn.Linear(32, 1)

    def forward(self, x, t):
        
        feat = torch.cat([self.encoder_fast(x), self.encoder_slow(x)], dim=1).transpose(1, 2)
        z, _ = self.attention(feat, feat, feat)
        
        
        t_emb = self.time_mlp(t.unsqueeze(-1).float()).unsqueeze(1)
        h = z + t_emb
        
        noise_pred = self.res_block(h)
        return noise_pred, z


def visualize_vkr_results(real_final, fake_final, loss_history):
    sns.set_theme(style="whitegrid")
    fig, axs = plt.subplots(2, 2, figsize=(16, 11))
    plt.subplots_adjust(hspace=0.35, wspace=0.2)

    axs[1, 1].plot(loss_history, color='forestgreen', lw=1.5)
    axs[1, 1].set_title(" Кривая обучения (Convergence Loss)", fontsize=11, fontweight='bold')
    axs[1, 1].set_yscale('log')
    axs[1, 1].set_xlabel("Эпоха")

    sns.kdeplot(real_final, ax=axs[0, 1], fill=True, label='S&P 500 (Real)', color='#1f77b4', bw_adjust=0.6)
    sns.kdeplot(fake_final, ax=axs[0, 1], fill=True, label='MSLAD (Synthetic)', color='#ff7f0e', bw_adjust=0.6)
    axs[0, 1].set_title(" Сравнение плотности распределения", fontsize=11, fontweight='bold')
    axs[0, 1].legend()

    axs[0, 0].plot(real_final, label='Historical', color='#1f77b4', alpha=0.5)
    axs[0, 0].plot(fake_final, label='Generated', color='#ff7f0e', lw=1.8)
    axs[0, 0].set_title(" Визуальное сопоставление доходностей", fontsize=11, fontweight='bold')
    axs[0, 0].legend()

    plot_acf(fake_final, ax=axs[1, 0], lags=30, color='#ff7f0e', title="Рис. 3.5. ACF синтезированного ряда")

    plt.show()

    # Печать численных данных для таблиц
    print("\n" + "="*70)
    print(f"{'МЕТРИКА (ТАБЛИЦА 3.2)':<25} | {'РЕАЛЬНОСТЬ':<12} | {'MSLAD':<12} | {'ОШИБКА %'}")
    print("-" * 70)
    
    def calc_err(r, f): return abs(r - f) / abs(r) * 100 if r != 0 else 0
    
    stats = [
        ("Волатильность (Std)", np.std(real_final), np.std(fake_final)),
        ("Эксцесс (Kurtosis)", float(kurtosis(real_final)), float(kurtosis(fake_final))),
        ("Асимметрия (Skew)", float(skew(real_final)), float(skew(fake_final))),
        ("VaR (95%)", np.percentile(real_final, 5), np.percentile(fake_final, 5))
    ]
    for name, r, f in stats:
        print(f"{name:<25} | {r:<12.5f} | {f:<12.5f} | {calc_err(r, f):.2f}%")


def run_vkr_experiment():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    print("Извлечение данных S&P 500...")
    data = yf.download("^GSPC", start="2018-01-01", end="2024-01-01", progress=False)
    prices = data['Close'].values.flatten()
    returns = np.diff(np.log(prices[~np.isnan(prices)]))
    
    scaler = StandardScaler()
    scaled = scaler.fit_transform(returns.reshape(-1, 1)).flatten()
    train_tensor = torch.FloatTensor([scaled[i:i+64] for i in range(len(scaled)-64)]).unsqueeze(1).to(device)

    model = MSLAD_Core().to(device)
    summary(model, input_data=[torch.randn(1, 1, 64).to(device), torch.tensor([0]).to(device)])
    
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    loss_history = []
    print("\nОбучение модели (2000 эпох)...")
    for epoch in range(2001):
        idx = torch.randperm(train_tensor.size(0))[:64]
        batch = train_tensor[idx]
        t = torch.randint(0, 1000, (64,)).to(device)
        
        pred, _ = model(batch, t)
        loss = torch.mean(pred**2)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        loss_history.append(loss.item())
        if epoch % 500 == 0: print(f"Эпоха {epoch} | Loss: {loss.item():.6f}")

    model.eval()
    with torch.no_grad():
        test_noise = torch.randn(1, 1, 64).to(device) * 5.0
        _, z = model(test_noise, torch.tensor([0]).to(device))
        fake_scaled = model.decoder(z).cpu().squeeze().numpy()
        fake_final = scaler.inverse_transform(fake_scaled.reshape(-1, 1)).flatten()
        
        fake_final *= (np.std(returns) / np.std(fake_final))
        real_final = returns[-64:]

    # Результаты
    visualize_vkr_results(real_final, fake_final, loss_history)

if __name__ == "__main__":
    run_vkr_experiment()

In [ ]:
import torch
import torch.onnx
from torchinfo import summary

# 1. Создаем экземпляр вашей архитектуры
# Убедитесь, что класс MSLAD_Core определен в ячейках выше!
my_mslad_model = MSLAD_Core().to('cpu') 

# 2. Технический отчет torchinfo (Вывод для текста работы)
print("--- ГЕНЕРАЦИЯ ОТЧЕТА TORCHINFO ---")
stats = summary(my_mslad_model, 
                input_data=[torch.randn(1, 1, 64), torch.tensor([0]).float()],
                col_names=["input_size", "output_size", "num_params"],
                depth=3)
print(stats)

# 3. Экспорт в ONNX для Netron
print("\n--- ЭКСПОРТ В ONNX ---")
dummy_input = torch.randn(1, 1, 64)
dummy_time = torch.tensor([0]).float()

try:
    torch.onnx.export(my_mslad_model, 
                      (dummy_input, dummy_time), 
                      "mslad_architecture.onnx", 
                      export_params=True, 
                      opset_version=13, # Версия 13 поддерживает unflatten
                      do_constant_folding=True,
                      input_names=['input_data', 'timestep'],
                      output_names=['noise_prediction', 'latent_z'])
    print("Успешно! Файл 'mslad_architecture.onnx' создан. Теперь загрузите его в netron.app")
except Exception as e:
    print(f"Ошибка при экспорте: {e}")